# World Happiness Report — Process Phase

Standardize the 5 yearly files (each with different column names and available fields),
reconcile country-name variants, backfill `region` for 2017-2019, and merge into one long-format
panel. Source: `data/raw/2015.csv` ... `2019.csv` (not committed — see `data/raw/README.md`).
Output: `data/processed/happiness_panel.parquet`.

## Step 1 — Load each year with its native columns

In [1]:
import pandas as pd
import os

RAW = "../data/raw"
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

raw = {y: pd.read_csv(os.path.join(RAW, f"{y}.csv")) for y in [2015, 2016, 2017, 2018, 2019]}
for y, df in raw.items():
    print(f"{y}: {df.shape}")

2015: (158, 12)
2016: (157, 13)
2017: (155, 12)
2018: (156, 9)
2019: (156, 9)

## Step 2 — Standardize column names

Each year uses different column names and drops different fields (see Prepare-phase table). Map
all 5 to one common `snake_case` schema.

In [2]:
col_maps = {
    2015: {"Country": "country", "Region": "region", "Happiness Rank": "rank",
           "Happiness Score": "happiness_score", "Economy (GDP per Capita)": "gdp_per_capita",
           "Family": "social_support", "Health (Life Expectancy)": "health_life_expectancy",
           "Freedom": "freedom", "Trust (Government Corruption)": "corruption_perception",
           "Generosity": "generosity", "Dystopia Residual": "dystopia_residual"},
    2016: {"Country": "country", "Region": "region", "Happiness Rank": "rank",
           "Happiness Score": "happiness_score", "Economy (GDP per Capita)": "gdp_per_capita",
           "Family": "social_support", "Health (Life Expectancy)": "health_life_expectancy",
           "Freedom": "freedom", "Trust (Government Corruption)": "corruption_perception",
           "Generosity": "generosity", "Dystopia Residual": "dystopia_residual"},
    2017: {"Country": "country", "Happiness.Rank": "rank", "Happiness.Score": "happiness_score",
           "Economy..GDP.per.Capita.": "gdp_per_capita", "Family": "social_support",
           "Health..Life.Expectancy.": "health_life_expectancy", "Freedom": "freedom",
           "Trust..Government.Corruption.": "corruption_perception", "Generosity": "generosity",
           "Dystopia.Residual": "dystopia_residual"},
    2018: {"Overall rank": "rank", "Country or region": "country", "Score": "happiness_score",
           "GDP per capita": "gdp_per_capita", "Social support": "social_support",
           "Healthy life expectancy": "health_life_expectancy",
           "Freedom to make life choices": "freedom", "Generosity": "generosity",
           "Perceptions of corruption": "corruption_perception"},
    2019: {"Overall rank": "rank", "Country or region": "country", "Score": "happiness_score",
           "GDP per capita": "gdp_per_capita", "Social support": "social_support",
           "Healthy life expectancy": "health_life_expectancy",
           "Freedom to make life choices": "freedom", "Generosity": "generosity",
           "Perceptions of corruption": "corruption_perception"},
}
std = {}
for y, df in raw.items():
    d = df.rename(columns=col_maps[y]).copy()
    d["year"] = y
    std[y] = d
print("Standardized. Example (2019):", [c for c in std[2019].columns if c != "year"])

Standardized. Example (2019): ['rank', 'country', 'happiness_score', 'gdp_per_capita', 'social_support', 'health_life_expectancy', 'freedom', 'generosity', 'corruption_perception']

## Step 3 — Reconcile country-name variants

Naming drifts between years for the *same* country (spelling, punctuation, official renames)
would otherwise silently break the multi-year panel.

In [3]:
rename_map = {
    "Macedonia": "North Macedonia",
    "North Cyprus": "Northern Cyprus",
    "Trinidad and Tobago": "Trinidad & Tobago",
    "Hong Kong S.A.R., China": "Hong Kong",
    "Taiwan Province of China": "Taiwan",
}
for y in std:
    before = set(std[y]["country"])
    std[y]["country"] = std[y]["country"].replace(rename_map)
    changed = before - set(std[y]["country"])
    if changed:
        print(f"{y}: renamed {sorted(changed)}")

country_sets = {y: set(d["country"]) for y, d in std.items()}
all_years = set.intersection(*country_sets.values())
union = set.union(*country_sets.values())
print(f"\nAfter reconciliation: {len(all_years)} countries in all 5 years (was 141 before), "
      f"{len(union)} total distinct (was 170 before)")

2015: renamed ['Macedonia', 'North Cyprus', 'Trinidad and Tobago']
2016: renamed ['Macedonia', 'North Cyprus', 'Trinidad and Tobago']
2017: renamed ['Hong Kong S.A.R., China', 'Macedonia', 'North Cyprus', 'Taiwan Province of China', 'Trinidad and Tobago']
2018: renamed ['Macedonia']

After reconciliation: 146 countries in all 5 years (was 141 before), 165 total distinct (was 170 before)

Remaining gaps (Angola, Djibouti, Oman, Somaliland region, Sudan, Suriname missing from
2019; Somalia, South Sudan, Gambia, Namibia added in later years) are **genuine list changes**,
not naming variants — left as-is.

## Step 4 — Backfill `region` for 2017-2019

In [4]:
region_map = {}
for y in [2015, 2016]:
    region_map.update(dict(zip(std[y]["country"], std[y]["region"])))
print(f"Built region map for {len(region_map)} countries from 2015/2016")

for y in [2017, 2018, 2019]:
    std[y]["region"] = std[y]["country"].map(region_map)
    missing = std[y]["region"].isna().sum()
    print(f"{y}: {missing} countries with no region match "
          f"({sorted(std[y].loc[std[y]['region'].isna(), 'country'].tolist())})")

Built region map for 164 countries from 2015/2016
2017: 0 countries with no region match ([])
2018: 0 countries with no region match ([])
2019: 1 countries with no region match (['Gambia'])

Gambia first appears in the 2019 file and was never in the 2015/2016 mapping source — its `region` is left null rather than guessed.

## Step 5 — Combine into one long-format panel

In [5]:
cols = ["country", "region", "year", "rank", "happiness_score", "gdp_per_capita",
        "social_support", "health_life_expectancy", "freedom", "corruption_perception",
        "generosity", "dystopia_residual"]
panel = pd.concat([std[y].reindex(columns=cols) for y in [2015, 2016, 2017, 2018, 2019]],
                   ignore_index=True)
print(f"Panel shape: {panel.shape}")
print(f"Countries: {panel['country'].nunique()}, Years: {sorted(panel['year'].unique())}")
print(f"Null happiness_score: {panel['happiness_score'].isna().sum()}")
print(f"Null corruption_perception: {panel['corruption_perception'].isna().sum()} (expected 1 - UAE 2018)")
print(f"Null dystopia_residual: {panel['dystopia_residual'].isna().sum()} "
      f"(expected {len(panel[panel['year'].isin([2018,2019])])} - not published for 2018/2019)")

Panel shape: (782, 12)
Countries: 165, Years: [2015, 2016, 2017, 2018, 2019]
Null happiness_score: 0
Null corruption_perception: 1 (expected 1 - UAE 2018)
Null dystopia_residual: 312 (expected 312 - not published for 2018/2019)

## Step 6 — Save the processed panel

In [6]:
panel.to_parquet(os.path.join(OUT_DIR, "happiness_panel.parquet"), index=False)
size_kb = os.path.getsize(os.path.join(OUT_DIR, "happiness_panel.parquet")) / 1024
print(f"Saved happiness_panel.parquet ({size_kb:.1f} KB)")

Saved happiness_panel.parquet (58.6 KB)

## Verification

The DuckDB SQL pipeline ([`sql/01_process_data.sql`](../sql/01_process_data.sql)) reproduces this
exactly (782 rows, 165 countries, 1 null `corruption_perception`) — but only after fixing a real
cross-engine discrepancy: DuckDB's CSV type inference does not treat the literal string `"N/A"`
(the UAE 2018 row) as NULL the way pandas does by default, so it silently inferred the whole
`corruption_perception` column as text. Fixed with an explicit `NULLIF(..., 'N/A')::DOUBLE` cast
in the SQL script — a good reminder that "the two pipelines agree" still needs the null-handling
checked, not just row counts.

## Summary

| Step | Result |
|---|---|
| Column standardization | 5 different native schemas → 1 common `snake_case` schema |
| Country-name reconciliation | 141/170 → **146/165** countries consistent across all years |
| Region backfill | 2017-2019 backfilled from 2015/2016; only Gambia (new in 2019) left null |
| Final panel | **782 rows, 165 countries, 5 years (2015-2019)** |